# Evaluating Arrays

Arrays need an **alignment** step before scoring: which gold element pairs with which
extracted element? The evaluator supports three strategies:

1. **Ordered** (default) -- Position matters. `"x-eval-align": {"ordered": true}`
2. **Key-field** -- match by a unique identifier field. Order doesn't matter.  `"x-eval-align": {"match_by": "key_field", "key": "name"}`
3. **Hungarian** -- optimal bipartite matching. Order doesn't matter, no key field needed, optimizes for best F1.     `"x-eval-align": {"match_by": "hungarian"}`

After alignment, each matched pair is scored recursively using the `items` schema.
Unmatched gold elements are **omissions**. Unmatched extracted elements are **hallucinations**.

## Data

Process steps where extracted has the right elements but in a **different order**,
plus realistic errors (wrong value, missing element, extra element).

In [1]:
GOLD = [
    # Record 0: one step, perfect match expected
    {"steps": [
        {"name": "deposit", "temp": 600},
    ]},
    # Record 1: two steps, extracted swaps order + wrong temp + extra step
    {"steps": [
        {"name": "deposit", "temp": 300},
        {"name": "anneal", "temp": 500},
    ]},
    # Record 2: three steps, extracted swaps order + missing step
    {"steps": [
        {"name": "deposit", "temp": 300},
        {"name": "anneal", "temp": 500},
        {"name": "clean", "temp": 100},
    ]},
]

EXTRACTED = [
    {"steps": [
        {"name": "deposit", "temp": 600},
    ]},
    {"steps": [
        {"name": "anneal", "temp": 480},    # swapped + wrong temp
        {"name": "deposit", "temp": 300},   # swapped but correct
        {"name": "cool", "temp": 50},       # hallucinated step
    ]},
    {"steps": [
        {"name": "anneal", "temp": 500},    # swapped but correct
        {"name": "deposit", "temp": 300},   # swapped but correct
        # "clean" is missing -> omission
    ]},
]

## Helper: display results

In [5]:
from struct_extract_eval import evaluate


def show_results(run):
    """Print per-record field results."""
    for record in run.records:
        print(f"\n--- Record {record.record_id}  "
              f"(F1: {record.f1:.3f}  P: {record.precision:.3f}  R: {record.recall:.3f}) ---")
        print(f"  {'Path':<25} {'Gold':<20} {'Extracted':<20} {'Score':>5}  {'Status'}")
        for fr in record.field_results:
            g = str(fr.gold_value)
            e = str(fr.extracted_value)
            print(f"  {fr.path:<25} {g:<20} {e:<20} {fr.score:>5.1f}  {fr.status}")

## Strategy 1: Ordered (default)

Pairs by position: No `x-eval-align` needed -- this is the default when the key is absent.

e.g. time series, ordered instructions.

In [6]:
ORDERED_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            # No x-eval-align -> ordered (positional) matching
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

run_ordered = evaluate(GOLD, EXTRACTED, schema=ORDERED_SCHEMA)
print(f"Ordered -- Mean F1: {run_ordered.mean_f1:.3f}")
show_results(run_ordered)

Ordered -- Mean F1: 0.333

--- Record 0  (F1: 1.000  P: 1.000  R: 1.000) ---
  Path                      Gold                 Extracted            Score  Status
  steps[0].name             deposit              deposit                1.0  match
  steps[0].temp             600                  600                    1.0  match

--- Record 1  (F1: 0.000  P: 0.000  R: 0.000) ---
  Path                      Gold                 Extracted            Score  Status
  steps[0].name             deposit              anneal                 0.0  mismatch
  steps[0].temp             300                  480                    0.0  mismatch
  steps[1].name             anneal               deposit                0.0  mismatch
  steps[1].temp             500                  300                    0.0  mismatch
  steps[-1].name            None                 cool                   0.0  hallucination
  steps[-1].temp            None                 50                     0.0  hallucination

--- Record 

Records 1 and 2 score poorly because the extractor swapped the element order.

**Note**: steps[-1] means this element in the array of extracted data is hallucination.

## Strategy 2: Key-Field Matching

Match elements by the value of a unique identifier field. Add `x-eval-align` to the
array node:

```json
"x-eval-align": {"match_by": "key_field", "key": "name"}
```

Gold's `"deposit"` pairs with extracted's `"deposit"`, regardless of position.

Use when elements have a natural unique key (name, id, formula).

In [9]:
KEYFIELD_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "x-eval-align": {"match_by": "key_field", "key": "name"},
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

run_keyfield = evaluate(GOLD, EXTRACTED, schema=KEYFIELD_SCHEMA)
print(f"Key-field -- Mean F1: {run_keyfield.mean_f1:.3f}")
show_results(run_keyfield)

Key-field -- Mean F1: 0.800

--- Record 0  (F1: 1.000  P: 1.000  R: 1.000) ---
  Path                      Gold                 Extracted            Score  Status
  steps[0].name             deposit              deposit                1.0  match
  steps[0].temp             600                  600                    1.0  match

--- Record 1  (F1: 0.600  P: 0.500  R: 0.750) ---
  Path                      Gold                 Extracted            Score  Status
  steps[0].name             deposit              deposit                1.0  match
  steps[0].temp             300                  300                    1.0  match
  steps[1].name             anneal               anneal                 1.0  match
  steps[1].temp             500                  480                    0.0  mismatch
  steps[-1].name            None                 cool                   0.0  hallucination
  steps[-1].temp            None                 50                     0.0  hallucination

--- Record 2  (F1:

Score improved. The swapped elements now match correctly by name:
- Record 1: `deposit` pairs with `deposit` (match), `anneal` pairs with `anneal` (name match, temp mismatch). `cool` has no gold counterpart -- hallucination.
- Record 2: `deposit` and `anneal` both match. `clean` has no extracted counterpart -- omission.

## Strategy 3: Hungarian Matching

When elements don't have a unique key field (e.g. arrays of primitives, or objects
without a natural ID), use Hungarian bipartite matching. It finds the optimal pairing
that maximizes total F1 across all pairs.

```json
"x-eval-align": {"match_by": "hungarian"}
```

Requires `scipy` (`pip install scipy`).

In [10]:
HUNGARIAN_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "x-eval-align": {"match_by": "hungarian"},
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

run_hungarian = evaluate(GOLD, EXTRACTED, schema=HUNGARIAN_SCHEMA)
print(f"Hungarian -- Mean F1: {run_hungarian.mean_f1:.3f}")
show_results(run_hungarian)

Hungarian -- Mean F1: 0.800

--- Record 0  (F1: 1.000  P: 1.000  R: 1.000) ---
  Path                      Gold                 Extracted            Score  Status
  steps[0].name             deposit              deposit                1.0  match
  steps[0].temp             600                  600                    1.0  match

--- Record 1  (F1: 0.600  P: 0.500  R: 0.750) ---
  Path                      Gold                 Extracted            Score  Status
  steps[0].name             deposit              deposit                1.0  match
  steps[0].temp             300                  300                    1.0  match
  steps[1].name             anneal               anneal                 1.0  match
  steps[1].temp             500                  480                    0.0  mismatch
  steps[-1].name            None                 cool                   0.0  hallucination
  steps[-1].temp            None                 50                     0.0  hallucination

--- Record 2  (F1:

Hungarian produces the same result as key-field here because the `name` field provides
enough signal for optimal matching. The difference matters when there's no obvious key.

## Other Exmaple: Hungarian with Primitive Arrays

In [12]:
TAGS_GOLD = [{"tags": ["silicon", "thin-film", "CVD"]}]
TAGS_EXTRACTED = [{"tags": ["CVD", "silicon", "PVD"]}]

TAGS_ORDERED = {
    "type": "object",
    "properties": {
        "tags": {
            "type": "array",
            "items": {"type": "string", "x-eval-compare": "exact"},
        },
    },
}

TAGS_HUNGARIAN = {
    "type": "object",
    "properties": {
        "tags": {
            "type": "array",
            "x-eval-align": {"match_by": "hungarian"},
            "items": {"type": "string", "x-eval-compare": "exact"},
        },
    },
}

run_tags_ord = evaluate(TAGS_GOLD, TAGS_EXTRACTED, schema=TAGS_ORDERED)
run_tags_hun = evaluate(TAGS_GOLD, TAGS_EXTRACTED, schema=TAGS_HUNGARIAN)

print(f"Ordered:   F1={run_tags_ord.mean_f1:.3f}  (positional: 'silicon' vs 'CVD' = mismatch)")
print(f"Hungarian: F1={run_tags_hun.mean_f1:.3f}  ('silicon' matches 'silicon', 'CVD' matches 'CVD')")
print()
show_results(run_tags_hun)

Ordered:   F1=0.000  (positional: 'silicon' vs 'CVD' = mismatch)
Hungarian: F1=0.667  ('silicon' matches 'silicon', 'CVD' matches 'CVD')


--- Record 0  (F1: 0.667  P: 0.667  R: 0.667) ---
  Path                      Gold                 Extracted            Score  Status
  tags[0]                   silicon              silicon                1.0  match
  tags[2]                   CVD                  CVD                    1.0  match
  tags[1]                   thin-film            None                   0.0  omission
  tags[-1]                  None                 PVD                    0.0  hallucination
